<a href="https://colab.research.google.com/github/anunyann/Tonalytics/blob/master/evaluate_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install sentence-transformers scikit-learn tqdm


Looking in indexes: https://download.pytorch.org/whl/cu121
INFO: pip is looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.5/780.5 MB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 64.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 55.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 107.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 13.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 5.4 MB/s e

In [2]:
from google.colab import files
uploaded = files.upload()

Saving tonalytics_files.zip to tonalytics_files.zip


In [3]:
import zipfile

with zipfile.ZipFile("tonalytics_files.zip", "r") as zip_ref:
    zip_ref.extractall(".")

In [4]:
import sys
sys.path.append("./src")

In [5]:
import json
import random
from tqdm import tqdm
import sys
sys.path.append("../src")

from final_recommender import HybridRecommender, load_data

In [6]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [7]:
from sentence_transformers import SentenceTransformer

sbert_model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')
print(sbert_model.device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

cpu


In [8]:
def train_test_split_playlist(playlist, min_tracks=5):
    """
    Split a playlist into observed tracks and hidden tracks.

    Args:
        playlist: original playlist dictionary
        min_tracks: minimum number of tracks required to split

    Returns:
        observed_tracks: list of tracks to be used for recommendation
        hidden_tracks: list of tracks to be predicted
    """
    if len(playlist['tracks']) < min_tracks:
        return None, None

    tracks = playlist['tracks'].copy()
    random.shuffle(tracks)
    split_idx = len(tracks) // 2

    observed = tracks[:split_idx]
    hidden = tracks[split_idx:]

    return observed, hidden

In [9]:
def recall_at_k(predicted_uris, hidden_tracks, k=100):
    """
    Compute Recall@k.

    Args:
        predicted_uris: list of predicted track URIs
        hidden_tracks: list of true hidden track dicts
        k: top-k predictions to consider

    Returns:
        recall value (float)
    """
    hidden_uris = set(track['track_uri'] for track in hidden_tracks)
    predicted_uris_at_k = set(predicted_uris[:k])

    hits = hidden_uris.intersection(predicted_uris_at_k)
    recall = len(hits) / len(hidden_uris) if hidden_uris else 0.0
    return recall

In [10]:
challenge_file = "challenge_set.json"
data = load_data(challenge_file)

train_playlists = []
test_playlists = []

for playlist in data['playlists']:
    observed, hidden = train_test_split_playlist(playlist)
    if observed and hidden:
        train_playlists.append({'tracks': observed, 'name': playlist.get('name', '')})
        test_playlists.append({
            'seed_tracks': observed,
            'hidden_tracks': hidden,
            'name': playlist.get('name', '')
        })

print(f"Training playlists: {len(train_playlists)}, Testing playlists: {len(test_playlists)}")


Loading data from challenge_set.json...
Training playlists: 8000, Testing playlists: 8000


In [11]:
model = HybridRecommender()
model.fit(train_playlists)

Training hybrid recommendation model...
Building collaborative filtering model...


100%|██████████| 8000/8000 [00:02<00:00, 3546.31it/s]


Building content-based model from playlist names...


100%|██████████| 8000/8000 [00:00<00:00, 103620.63it/s]


Batches:   0%|          | 0/184 [00:00<?, ?it/s]

Model training complete!


In [ ]:
recalls = []
test_playlists = test_playlists[:1000] # imav staveno na site 8000 ama trebase da cekam 3 saati.. ne znaev sto e podobro pa rekov deka na site 8000 ke pravam samo na finalnoto
for playlist in tqdm(test_playlists, desc="Evaluating playlists"):
    seed_tracks = [track['track_uri'] for track in playlist['seed_tracks']]
    hidden_tracks = playlist['hidden_tracks']
    playlist_name = playlist.get('name', None)

    # Generate recommendations
    recommended = model.recommend(
        seed_tracks=seed_tracks,
        playlist_name=playlist_name,
        n=500
    )

    # Calculate Recall@100
    recall = recall_at_k(recommended, hidden_tracks, k=100)
    recalls.append(recall)

# Final result
average_recall = sum(recalls) / len(recalls)
print(f"\n Average Recall@100: {average_recall:.4f}")

Evaluating playlists: 100%|██████████| 1000/1000 [37:18<00:00,  2.24s/it]


 Average Recall@100: 0.0160


In [ ]:
popularity_weights = [0.0, 0.05, 0.1, 0.2]
name_weights = [1, 3, 5, 10]
# Store results
results = []
test_playlists = test_playlists[:1000]

# Grid search
for pop_w in popularity_weights:
    for name_w in name_weights:
        print(f"\nTesting: popularity_weight={pop_w}, name_weight={name_w}")

        # Create a fresh model for each combination
        model = HybridRecommender()
        model.fit(train_playlists)
        model.popularity_weight = pop_w
        model.name_weight = name_w

        # Evaluate
        recalls = []
        for playlist in tqdm(test_playlists, desc="Evaluating playlists", leave=False):
            seed_tracks = [track['track_uri'] for track in playlist['seed_tracks']]
            hidden_tracks = playlist['hidden_tracks']
            playlist_name = playlist.get('name', None)

            recommended = model.recommend(
                seed_tracks=seed_tracks,
                playlist_name=playlist_name,
                n=500
            )

            recall = recall_at_k(recommended, hidden_tracks, k=100)
            recalls.append(recall)

        avg_recall = sum(recalls) / len(recalls)
        print(f"Average Recall@100: {avg_recall:.4f}")

        # Store the result
        results.append({
            'popularity_weight': pop_w,
            'name_weight': name_w,
            'recall_at_100': avg_recall
        })

print("\n Grid search complete!")


Testing: popularity_weight=0.0, name_weight=1
Training hybrid recommendation model...
Building collaborative filtering model...


100%|██████████| 8000/8000 [00:02<00:00, 3810.00it/s] 


Building content-based model from playlist names...


100%|██████████| 8000/8000 [00:00<00:00, 128265.20it/s]


Batches:   0%|          | 0/184 [00:00<?, ?it/s]

...

Building content-based model from playlist names...


100%|██████████| 8000/8000 [00:00<00:00, 82897.12it/s]


Batches:   0%|          | 0/184 [00:00<?, ?it/s]

Model training complete!


Evaluating playlists:  52%|█████▏    | 515/1000 [11:46<11:22,  1.41s/it]